<a href="https://colab.research.google.com/github/duckmaster168/Research_MPGELU_Testing_Students/blob/main/Test_2(plus%20Lambdas%20and%20GELUs%20data).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Environment Setup & Core Functions**

In [5]:
import os
import torch
import torch.nn as nn
import pandas as pd
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torchvision import datasets

# Local project modules
from DataTransforms import DataTransforms
from MyCNN import MyCNN
from Trainer import Trainer
from activations import get_activation

# Select hardware accelerator
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Executing experiments on device: {device}")

# Classification accuracy metric
def accuracy_fn(y_true, y_pred):
    correct = torch.eq(y_true, y_pred).sum().item()
    return (correct / len(y_pred)) * 100.0

# Directory for persistent records
checkpoint_dir = "experiment_checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)

Executing experiments on device: cpu


# **Data Pipeline Initialization**


In [ ]:
# Initialize data pipelines with normalization and augmentation
transforms_handler = DataTransforms(dataset="cifar10", use_augmentation=True, use_stats=True)

train_dataset = datasets.CIFAR10(
    root="./data", train=True, download=True, transform=transforms_handler.get_train_transform()
)
test_dataset = datasets.CIFAR10(
    root="./data", train=False, download=True, transform=transforms_handler.get_test_transform()
)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=2, pin_memory=True)

print(f"Dataset ready | Training samples: {len(train_dataset)} | Test samples: {len(test_dataset)}")

 49%|████▉     | 83.9M/170M [17:25<15:50, 91.1kB/s]

# **Execution Loop Across All 6 Baselines with Auto-Saving**

In [ ]:
# CNN architecture matching 3x32 -> 2x64 -> 2x128 configuration
cnn_arch = [32, 32, 32, "MaxPool", 64, 64, "MaxPool", 128, 128, "MaxPool"]
baselines = ["relu", "leakyrelu", "gelu", "pgelu", "lambdagelu", "mpgelu"]

epochs = 15
results = {}

for name in baselines:
    print(f"\n=================== Training Baseline: {name.upper()} ===================")
    act_class = get_activation(name)

    # Instantiate model
    model = MyCNN(input_shape=3, output_shape=10, activation=act_class, params=cnn_arch).to(device)
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    trainer = Trainer(
        model=model,
        loss_fn=loss_fn,
        optimizer=optimizer,
        calculate_accuracy=accuracy_fn,
        device=device,
        loss_steps=1
    )

    history = {
        "train_loss": [], "train_acc": [],
        "test_loss": [], "test_acc": [],
        "grad_norms": [], "lambdas": []
    }

    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc, grads, lams = trainer.train(train_loader, epoch=epoch)
        te_loss, te_acc = trainer.test(test_loader, epoch=epoch)

        history["train_loss"].append(tr_loss)
        history["train_acc"].append(tr_acc)
        history["test_loss"].append(te_loss)
        history["test_acc"].append(te_acc)
        history["grad_norms"].append(grads)
        history["lambdas"].append(lams)

    results[name] = history

    # Save weights and run history to disk (.pth)
    checkpoint_payload = {
        "activation_name": name,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "history": history,
        "epochs": epochs
    }
    save_path = os.path.join(checkpoint_dir, f"baseline_{name}.pth")
    torch.save(checkpoint_payload, save_path)
    print(f"Saved checkpoint & metric history: {save_path}")

# **Plot & Save Accuracy and Loss Curves**

In [ ]:
plt.figure(figsize=(15, 5))

# Test Accuracy Subplot
plt.subplot(1, 2, 1)
for name, hist in results.items():
    plt.plot(range(1, epochs + 1), hist["test_acc"], label=name.upper(), linewidth=2)
plt.title("CIFAR-10 Test Accuracy Comparison", fontsize=12, fontweight="bold")
plt.xlabel("Epochs")
plt.ylabel("Accuracy (%)")
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend()

# Test Loss Subplot
plt.subplot(1, 2, 2)
for name, hist in results.items():
    plt.plot(range(1, epochs + 1), hist["test_loss"], label=name.upper(), linewidth=2)
plt.title("CIFAR-10 Test Loss Comparison", fontsize=12, fontweight="bold")
plt.xlabel("Epochs")
plt.ylabel("Cross Entropy Loss")
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend()

plt.tight_layout()
plt.savefig(os.path.join(checkpoint_dir, "cifar10_accuracy_loss_curves.png"), dpi=300, bbox_inches="tight")
plt.show()

# **Plot & Save Layer-Wise Gradient Norms**

In [ ]:
plt.figure(figsize=(10, 5))

for name in baselines:
    grad_history = results[name]["grad_norms"]
    if grad_history and len(grad_history[-1]) > 0:
        final_norms = list(grad_history[-1].values())
        mean_grad = sum(final_norms) / len(final_norms)
        plt.bar(name.upper(), mean_grad, alpha=0.85)

plt.title("Mean Layer-Wise Gradient Norm ($L_2$) at Final Epoch", fontsize=12, fontweight="bold")
plt.ylabel("Gradient Norm Magnitude")
plt.grid(axis="y", linestyle="--", alpha=0.6)

plt.tight_layout()
plt.savefig(os.path.join(checkpoint_dir, "cifar10_gradient_norms.png"), dpi=300, bbox_inches="tight")
plt.show()

# **Plot & Save Parameter Trajectory ($\lambda$ Drift)**

In [ ]:
plt.figure(figsize=(10, 5))

for name in ["lambdagelu", "mpgelu"]:
    if name in results:
        lambdas = results[name]["lambdas"]
        if lambdas and len(lambdas[0]) > 0:
            avg_lambdas = [sum(l) / len(l) for l in lambdas]
            plt.plot(range(1, epochs + 1), avg_lambdas, label=f"{name.upper()} (Mean $\lambda$)", linewidth=2, marker="o")

plt.axhline(y=1.0, color="r", linestyle=":", label="Lower Bound Constraint ($\lambda = 1.0$)")
plt.title("Sharpness Parameter Trajectory ($\lambda$ Drift) During Training", fontsize=12, fontweight="bold")
plt.xlabel("Epochs")
plt.ylabel("Lambda Parameter ($\lambda$)")
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend()

plt.tight_layout()
plt.savefig(os.path.join(checkpoint_dir, "cifar10_lambda_drift.png"), dpi=300, bbox_inches="tight")
plt.show()

# **Quantitative Performance Summary Table**

In [ ]:
summary_metrics = []

for name in baselines:
    file_path = os.path.join(checkpoint_dir, f"baseline_{name}.pth")
    if os.path.exists(file_path):
        chkpt = torch.load(file_path, map_location="cpu")
        hist = chkpt["history"]

        summary_metrics.append({
            "Activation": name.upper(),
            "Final Test Acc (%)": round(hist["test_acc"][-1], 2),
            "Best Test Acc (%)": round(max(hist["test_acc"]), 2),
            "Final Test Loss": round(hist["test_loss"][-1], 4),
            "Min Test Loss": round(min(hist["test_loss"]), 4),
            "Saved Path": file_path
        })

summary_df = pd.DataFrame(summary_metrics)
summary_df